In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
# from consensus_aligner import ChromatographicAligner
from ipyfilechooser import FileChooser
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess, sys, threading, os

In [ ]:
class ChromatographicAlignerUI(Interface):
    def __init__(self):
        super().__init__(supported_extensions=('.txt',))
        self._setup_default_parameters() 
        self._create_parameter_widgets()
        self._create_base_widgets()
        self._create_action_widgets()
        self._create_alignment_widget()
        self._setup_callbacks()
        self._setup_environment()
        self._create_optional_filter_widgets()

        # états
        self.align_state = "idle"         # "idle" | "running" | "ready" | "error"
        self.has_alignment_results = False

    def _setup_default_parameters(self):
        """Set up default parameters for the chromatographic alignment."""
        #public configurable
        self.rt1_penalty = "1"
        self.rt2_penalty = "5"
        self.similarity_cutoff = "90"
        self.missing_value_limit = 0.05 # MON FILTER

        # private, fixed
        self._disimilarity_cutoff = 90
        self._num_cores = 1
        self._quant_method= "T"
        self._auto_tune_match_stringency = False
        self._missing_peak_finder_similarity_lax = 0.85

    
    def _create_parameter_widgets(self):
        self.w_seedFile = widgets.Text(value='1')
        self.seedFile = self._bold_widget("Seed file", self.w_seedFile)
        self.seed_def = self.create_help_text(
             "File number in inputFileList to initialize alignment."
        )
        self.w_rt1_penalty = widgets.Text(value=self.rt1_penalty)
        self.rt1_penalty = self._bold_widget("RT1 Penalty", self.w_rt1_penalty)
        self.rt1_penalty_def = self.create_help_text(
            "Penalty used for first retention time errors.  Defaults to 1."
        )
        self.w_rt2_penalty = widgets.Text(value=self.rt2_penalty)
        self.rt2_penalty = self._bold_widget("RT2 Penalty", self.w_rt2_penalty)
        self.rt2_penalty_def = self.create_help_text(
            "Penalty used for second retention time errors. Defaults to 5."
        )
        self.w_similarity_cutoff = widgets.Text(value=self.similarity_cutoff)
        self.similarity_cutoff = self._bold_widget("Similarity Cutoff", self.w_similarity_cutoff)
        self.similarity_cutoff_def = self.create_help_text(
            "Adjusts peak similarity threshold required for alignment."
            "Adjust in concordance with RT1 and RT2 penalties. " \
            "Will be ignored if autoTuneMatchStrigency is TRUE. Defaults to 90."
            )
        
    def _create_alignment_widget(self):
        self.txt_title = widgets.HTML(value="<H1>Chromatographic Alignment</H1>")
         # NIST matching
        self.nist = widgets.Checkbox(
            value=True,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        )
              # Action widgets
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()


    def _on_button_click(self, b):
        """Handle button click event."""

        with self.output:
            self.output.clear_output()
            print("Running alignment... ")
            # validate parameter #TODO
            # errors = self._validate_parameters()
            if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
                print("Output directory cannot be empty")
                return
            
            print("\n Collecting files from selections...   ")
            selected_files = self.get_all_files_from_selections()
            if not selected_files:
                print("Please select files or folders containing .txt files.")
                return
            print(f"\n✅ {len(selected_files)} compatible files found")
            for i, f in enumerate(selected_files, 1):
                print(f"  {i}. {selected_files [i-1]}")
            print(f"{'='*60}")
            
            seed_file= int(self.w_seedFile.value.strip()) -1
            # 👉 démarre : en cours, pas de résultats encore
            self.align_state = "running"
            self.has_alignment_results = False
            
            self._start_subprocess_alignment(selected_files, str(seed_file))


    def _start_subprocess_alignment(self, selected_files, seed_file):
        """Start the alignment process in a subprocess."""
        
        alignment_params = [
            sys.executable,
            '/app/src/peak_alignment_cli.py',
            '--seed_file', seed_file,
            '--output_path', self.get_output_path(),
            '--rt1_penalty', self.w_rt1_penalty.value,
            '--rt2_penalty', self.w_rt2_penalty.value,
            '--similarity_cutoff', self.w_similarity_cutoff.value,
            '--disimilarity_cutoff', (self._disimilarity_cutoff),
            '--num_cores', (self._num_cores),
            '--missing_value_limit', (self.missing_value_limit),
            '--quant_method', self._quant_method,
            '--missing_peak_finder_similarity_lax', (self._missing_peak_finder_similarity_lax)
        ]
        # cas des booleens
        if self._auto_tune_match_stringency:
            alignment_params.append('--auto_tune_match_stringency')
        # cas des listes
        alignment_params +=["--input"] + selected_files
        
        alignment_params= list(map(str, alignment_params))

        self.current_process = subprocess.Popen(
            alignment_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        self.stop_button.disabled = False

        def stream_output(proc, output_widget):
            try:
                for line in iter(proc.stdout.readline, ''):
                    with output_widget:
                        print(line, end='')
                    # Vérifie si le process est terminé après chaque ligne
                    if proc.poll() is not None:
                        break
            except Exception as e:
                with output_widget:
                    print(f"⚠️ Error reading process output: {e}")
            finally:
                proc.stdout.close()
                retcode = proc.wait()
                with output_widget:
                    if retcode == 0:
                        print("\n✅ Analyse terminée avec succès")
                        print(f"{'='*60}")
                         # 👉 terminé avec succès
                        self.align_state = "ready"
                        self.has_alignment_results = True
                    else:
                        print(f"\n❌ L'analyse a échoué avec le code de retour {retcode}")
                        print(f"\n{'='*60}")
                        # 👉 terminé en erreur
                        self.align_state = "error"
                        self.has_alignment_results = False
                self.stop_button.disabled = True
                self.current_process = None

        # thread pour afficher stdout en direct
        threading.Thread(
            target=stream_output,
            args=(self.current_process,
                  self.output),
                  daemon=True
            ).start()
        
    def _create_optional_filter_widgets(self):
        self.w_new_missing_value_limit = widgets.Text(value="0.5")
        self.new_missing_value_limit = self._bold_widget("Missing Value Limit", self.w_new_missing_value_limit)
        self.missing_value_limit_def = self.create_help_text(
            "Maximum fraction (Numeric between 0 and 1) of missing values acceptable \
                for retaining a metabolite in the final alignment table. Defaults to 0.05 in \
                    chromatographic alignment. Defaults to 0.5 in optional post-processing."
        )
        self.apply_filter_button = widgets.Button(
            description='Apply Filter',
            button_style='info',
            icon='filter',
            style=self.style,
            disabled=False
        )
        self.apply_filter_button.on_click(self._on_filter_button_click)
    
    def _on_filter_button_click(self, b):
        """Handle filter button click event."""
        with self.output:
            self.output.clear_output()
            if self.align_state == "running":
                print("⏳ Alignment is still running. Please wait until it finishes.")
                print(f"\n{'='*60}")
                return
            # 2) pas de résultats prêts ?
            if not self.has_alignment_results or self.align_state != "ready":
                print("❌ No alignment results found. Please run alignment first.")
                print(f"\n{'='*60}")
                return
            # 3) OK, on peut lancer le filtrage
            print("✅ Alignment results detected. Starting filtering...")
            self._start_subprocess_alignment_filter()
    
    def _start_subprocess_alignment_filter(self):
        filter_params = [
            sys.executable,
            "/app/src/peak_alignment_filter_cli.py",
            '--new_missing_value_limit', self.w_new_missing_value_limit.value,
            '--output_path', self.get_output_path(),
            '--rt1_penalty', self.w_rt1_penalty.value,
            '--rt2_penalty', self.w_rt2_penalty.value,
            '--similarity_cutoff', self.w_similarity_cutoff.value,
            '--disimilarity_cutoff', (self._disimilarity_cutoff),
            '--num_cores', (self._num_cores),
            '--missing_value_limit', (self.missing_value_limit),
            '--quant_method', self._quant_method,
            '--missing_peak_finder_similarity_lax', (self._missing_peak_finder_similarity_lax)
        ]
        # cas des booleens
        if self._auto_tune_match_stringency:
            filter_params.append('--auto_tune_match_stringency')
        filter_params = list(map(str, filter_params))
        

        self.current_process = subprocess.Popen(
            filter_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        self.stop_button.disabled = False
        def stream_output(proc, output_widget):
            try:
                for line in iter(proc.stdout.readline, ''):
                    with output_widget:
                        print(line, end='')
                    # Vérifie si le process est terminé après chaque ligne
                    if proc.poll() is not None:
                        break
            except Exception as e:
                with output_widget:
                    print(f"⚠️ Error reading process output: {e}")
            finally:
                proc.stdout.close()
                retcode = proc.wait()
                with output_widget:
                    if retcode == 0:
                        print("\n✅ Analyse terminée avec succès")
                        print(f"{'='*60}")
                    else:
                        print(f"\n❌ L'analyse a échoué avec le code de retour {retcode}")
                        print(f"\n{'='*60}")
                self.stop_button.disabled = True
                self.current_process = None

        # thread pour afficher stdout en direct
        threading.Thread(
            target=stream_output,
            args=(self.current_process,
                  self.output),
                  daemon=True
            ).start()

    def display(self):
        """Display the interface."""
        display(self.txt_title,
                widgets.VBox([self._vbox, self._vbox2]),
                self.seedFile,
                self.seed_def,
                self.rt1_penalty,
                self.rt1_penalty_def,
                self.rt2_penalty,
                self.rt2_penalty_def,
                self.similarity_cutoff,
                self.similarity_cutoff_def,
                self.nist,
                widgets.HBox([self.run_button, self.stop_button, self.clear_button]),
                widgets.HTML("<hr><b>Optional Post-processing</b>"),
                self.new_missing_value_limit,
                self.missing_value_limit_def,
                self.apply_filter_button,
                self.output)
    


In [ ]:
t = ChromatographicAlignerUI()
t.display()